In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate

import torch

In [2]:
ag = [[4], [2,5], [1,2], [4, 7], [4,5,7], [0,1,4], [7,8], [6,7], [3,4, 6]]
gate_order = [8,5,2,1,4,7,6,3,0]

def load_unitary_gates():
    gates = []
    for i in gate_order:
        # 注意：这里需要把 torch tensor 转为 numpy array 以便后续处理
        gate = torch.load(f'../gates/unitary_gates/tensor{i}.pt').detach().numpy()
        gate = gate.reshape(-1, 2**len(ag[i]))
        gates.append(gate)
    return gates

In [3]:
qc = QuantumCircuit(9)
gates = load_unitary_gates()
# gates[0]
apply_list = [ag[i] for i in gate_order]
for gate_matrix, qubits in zip(gates, apply_list):
        # 使用 Qiskit 的 UnitaryGate 包装你的矩阵
        print(gate_matrix.shape, qubits)
        custom_gate = UnitaryGate(gate_matrix)
        qc.append(custom_gate, qubits[::-1])
    # 添加全测量
qc.measure_all()

(8, 8) [3, 4, 6]
(8, 8) [0, 1, 4]
(4, 4) [1, 2]
(4, 4) [2, 5]
(8, 8) [4, 5, 7]
(4, 4) [6, 7]
(4, 4) [7, 8]
(4, 4) [4, 7]
(2, 2) [4]


In [4]:
## 经典模拟器的结果
from qiskit_aer import AerSimulator
from qiskit import transpile

# 选择模拟器
backend = AerSimulator()

# 编译并运行
t_qc = transpile(qc, backend)
job = backend.run(t_qc, shots=4096)
result = job.result()

counts = result.get_counts()
print(counts)

{'001100100': 1383, '110010000': 1342, '000001011': 1371}


In [5]:
import qiskit.qasm2 as qasm2
# 使用夸父进行实验：
transpiled_qc = transpile(qc, basis_gates=['h','rx', 'ry', 'rz', 'cz'], optimization_level=3)
qasm2.dump(transpiled_qc, open('../gates/transpiled_circuit.qasm', 'w'))
